In [1]:
import pandas as pd
import numpy as np
import os

branches_df = pd.read_csv('/content/branches_clean.csv')

customers_df = pd.read_csv('/content/customers_clean.csv')

inventory_df = pd.read_csv('/content/inventory_clean.csv')

invoices_df = pd.read_csv('/content/invoices_clean.csv')

payments_df = pd.read_csv('/content/payments_clean.csv')

products_df = pd.read_csv('/content/products_clean.csv')

purchase_orders_header_df = pd.read_csv('/content/purchase_headers_clean.csv')

purchase_orders_lines_df = pd.read_csv('/content/purchase_lines_clean.csv')

sales_orders_header_df = pd.read_csv('/content/sales_headers_clean.csv')

sales_orders_lines_df = pd.read_csv('/content/sales_lines_clean.csv')

stock_ledger_df = pd.read_csv('/content/stock_ledger_clean.csv')

suppliers_df = pd.read_csv('/content/suppliers_clean.csv')

**Load Integrated Datasets**

In [2]:
sales_integrated_df = pd.read_csv('/content/sales_integrated.csv')

purchase_integrated_df = pd.read_csv('/content/purchase_integrated.csv')

print(
    "Sales integrated:",
    sales_integrated_df.shape
)

print(
    "Purchase integrated:",
    purchase_integrated_df.shape
)

Sales integrated: (130402, 55)
Purchase integrated: (155495, 51)


### 1. Validate row counts

In [3]:
validation_rows = pd.DataFrame({

    'Dataset': [
        'Sales Header',
        'Sales Lines',
        'Integrated Sales',
        'Purchase Header',
        'Purchase Lines',
        'Integrated Purchases'
    ],

    'Rows': [
        len(sales_orders_header_df),
        len(sales_orders_lines_df),
        len(sales_integrated_df),
        len(purchase_orders_header_df),
        len(purchase_orders_lines_df),
        len(purchase_integrated_df)
    ]

})

display(validation_rows)

,Dataset,Rows
0,Sales Header,20000
1,Sales Lines,130402
2,Integrated Sales,130402
3,Purchase Header,24000
4,Purchase Lines,155495
5,Integrated Purchases,155495


### **2. Check Duplicates**

In [4]:
validation_duplicates = []

validation_datasets = {

    'Sales Header':
        sales_orders_header_df,

    'Sales Lines':
        sales_orders_lines_df,

    'Integrated Sales':
        sales_integrated_df,

    'Purchase Header':
        purchase_orders_header_df,

    'Purchase Lines':
        purchase_orders_lines_df,

    'Integrated Purchases':
        purchase_integrated_df
}

for name, df in validation_datasets.items():

    duplicate_count = df.duplicated().sum()

    validation_duplicates.append({

        'Dataset': name,

        'Rows': len(df),

        'Duplicate Rows':
            duplicate_count,

        'Duplicate %':
            round(
                duplicate_count
                / len(df)
                * 100,
                2
            )

    })

validation_duplicates_df = pd.DataFrame(
    validation_duplicates
)

display(validation_duplicates_df)

,Dataset,Rows,Duplicate Rows,Duplicate %
0,Sales Header,20000,0,0.0
1,Sales Lines,130402,0,0.0
2,Integrated Sales,130402,0,0.0
3,Purchase Header,24000,0,0.0
4,Purchase Lines,155495,0,0.0
5,Integrated Purchases,155495,0,0.0


### **3. Check Missing Values**

In [5]:
validation_missing = []

for name, df in validation_datasets.items():

    missing_count = df.isna().sum().sum()

    total_cells = (
        df.shape[0]
        * df.shape[1]
    )

    validation_missing.append({

        'Dataset': name,

        'Missing Cells':
            missing_count,

        'Missing %':
            round(
                missing_count
                / total_cells
                * 100,
                2
            )

    })

validation_missing_df = pd.DataFrame(
    validation_missing
)

display(validation_missing_df)

,Dataset,Missing Cells,Missing %
0,Sales Header,0,0.00
1,Sales Lines,0,0.00
2,Integrated Sales,0,0.00
3,Purchase Header,2370,0.99
4,Purchase Lines,0,0.00
5,Integrated Purchases,15418,0.19


### **4. Validate Potential Primary Keys**

In [6]:
def validate_key(
    df,
    key,
    dataset_name
):

    if key not in df.columns:

        print(
            f"{key} does not exist "
            f"in {dataset_name}"
        )

        return

    print(
        f"\n{dataset_name} — {key}"
    )

    print(
        "Rows:",
        len(df)
    )

    print(
        "Unique values:",
        df[key].nunique(
            dropna=True
        )
    )

    print(
        "Missing:",
        df[key].isna().sum()
    )

    print(
        "Duplicate:",
        df[key].duplicated().sum()
    )

### **5. Validate Important IDs**

In [7]:
validate_key(
    products_df,
    'product_id',
    'Products'
)

validate_key(
    customers_df,
    'customer_id',
    'Customers'
)

validate_key(
    suppliers_df,
    'supplier_id',
    'Suppliers'
)

validate_key(
    sales_orders_header_df,
    'order_id',
    'Sales Header'
)


Products — product_id
Rows: 30
Unique values: 30
Missing: 0
Duplicate: 0

Customers — customer_id
Rows: 500
Unique values: 500
Missing: 0
Duplicate: 0

Suppliers — supplier_id
Rows: 8
Unique values: 8
Missing: 0
Duplicate: 0
order_id does not exist in Sales Header


### **6. Referential Integrity - Sales**

In [11]:
if (
    'order_id' in sales_orders_lines_df.columns
    and
    'order_id' in sales_orders_header_df.columns
):

    unmatched_orders = (
        sales_orders_lines_df[
            ~sales_orders_lines_df[
                'order_id'
            ].isin(
                sales_orders_header_df[
                    'order_id'
                ]
            )
        ]
    )

    print(
        "Sales lines without "
        "matching order:",
        len(unmatched_orders)
    )

### **7. Referential Integrity - Products**

In [10]:
if (
    'product_id' in sales_orders_lines_df.columns
    and
    'product_id' in products_df.columns
):

    unmatched_products = (
        sales_orders_lines_df[
            ~sales_orders_lines_df[
                'product_id'
            ].isin(
                products_df[
                    'product_id'
                ]
            )
        ]
    )

    print(
        "Sales lines without "
        "matching product:",
        len(unmatched_products)
    )

Sales lines without matching product: 0


### **8. Referential Integrity - Customers**

In [12]:
if (
    'customer_id' in sales_orders_header_df.columns
    and
    'customer_id' in customers_df.columns
):

    unmatched_customers = (
        sales_orders_header_df[
            ~sales_orders_header_df[
                'customer_id'
            ].isin(
                customers_df[
                    'customer_id'
                ]
            )
        ]
    )

    print(
        "Sales orders without "
        "matching customer:",
        len(unmatched_customers)
    )

Sales orders without matching customer: 0


### **9. Validate Numeric Values**

In [13]:
numeric_validation = []

for name, df in validation_datasets.items():

    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    for column in numeric_columns:

        numeric_validation.append({

            'Dataset': name,

            'Column': column,

            'Minimum':
                df[column].min(),

            'Maximum':
                df[column].max(),

            'Mean':
                df[column].mean(),

            'Negative Values':
                (df[column] < 0).sum(),

            'Zero Values':
                (df[column] == 0).sum()

        })

numeric_validation_df = pd.DataFrame(
    numeric_validation
)

display(numeric_validation_df)

,Dataset,Column,Minimum,Maximum,Mean,Negative Values,Zero Values
0,Sales Header,total_order_value,310.0,8169760.0,1.272370e+06,0,0
1,Sales Header,total_gst_amount,55.8,2282221.8,3.488681e+05,0,0
2,Sales Header,grand_total,365.8,10451981.8,1.621238e+06,0,0
3,Sales Lines,line_number,1.0,12.0,4.670672e+00,0,0
4,Sales Lines,quantity,1.0,20.0,1.049473e+01,0,0
...,...,...,...,...,...,...,...
66,Integrated Purchases,reorder_level,3.0,200.0,3.967688e+01,0,0
67,Integrated Purchases,safety_stock,2.0,100.0,2.010687e+01,0,0
68,Integrated Purchases,max_stock_level,8.0,500.0,1.047148e+02,0,0
69,Integrated Purchases,lead_time_days_product,4.0,40.0,1.700226e+01,0,0


### **10. Validate Dates**

In [14]:
date_validation = []

for name, df in validation_datasets.items():

    for column in df.columns:

        if (
            'date' in column.lower()
            or
            'datetime' in column.lower()
        ):

            dates = pd.to_datetime(
                df[column],
                errors='coerce'
            )

            date_validation.append({

                'Dataset': name,

                'Column': column,

                'Minimum Date':
                    dates.min(),

                'Maximum Date':
                    dates.max(),

                'Invalid Dates':
                    dates.isna().sum()

            })

date_validation_df = pd.DataFrame(
    date_validation
)

display(date_validation_df)

,Dataset,Column,Minimum Date,Maximum Date,Invalid Dates
0,Sales Header,order_date,2019-01-01,2024-12-31,0
1,Sales Header,delivery_date,2019-01-02,2025-01-13,0
2,Integrated Sales,order_date,2019-01-01,2024-12-31,0
3,Integrated Sales,delivery_date,2019-01-02,2025-01-13,0
4,Integrated Sales,last_purchase_date,2019-03-15,2025-04-27,0
5,Integrated Sales,last_purchase_date_customer,2015-04-30,2024-12-27,0
6,Purchase Header,order_date,2019-01-01,2024-12-31,0
7,Purchase Header,expected_delivery_date,2019-01-11,2025-01-28,0
8,Purchase Header,received_date,2019-01-11,2025-01-28,2370
9,Integrated Purchases,order_date,2019-01-01,2024-12-31,0


### **11. Check integrated sales data**

In [15]:
print(
    "Integrated Sales Shape:",
    sales_integrated_df.shape
)

display(
    sales_integrated_df.head()
)

print("\nMissing values:")

display(
    sales_integrated_df.isna()
    .sum()
    .sort_values(
        ascending=False
    )
    .head(20)
)

Integrated Sales Shape: (130402, 55)


,so_id,line_number,product_id,quantity,unit_price,gst_rate,line_total,gst_amount,line_grand_total,customer_id,...,pincode,region,branch_id_customer,credit_limit,current_balance,payment_terms_customer,customer_since,last_purchase_date_customer,total_purchase_value,customer_rating
0,SO-990591,1,P018,19,11300,28,214700,60116.0,274816.0,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
1,SO-990591,2,P026,2,310,18,620,111.6,731.6,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
2,SO-990591,3,P026,3,310,18,930,167.4,1097.4,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
3,SO-990591,4,P013,9,1450,18,13050,2349.0,15399.0,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
4,SO-990591,5,P004,3,39800,28,119400,33432.0,152832.0,C0070,...,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3



Missing values:


,0
so_id,0
line_number,0
product_id,0
quantity,0
unit_price,0
gst_rate,0
line_total,0
gst_amount,0
line_grand_total,0
customer_id,0


### **12. Check whether integration duplicated rows**

In [16]:
original_sales_rows = len(
    sales_orders_lines_df
)

integrated_sales_rows = len(
    sales_integrated_df
)

print(
    "Original sales-line rows:",
    original_sales_rows
)

print(
    "Integrated sales rows:",
    integrated_sales_rows
)

print(
    "Difference:",
    integrated_sales_rows
    - original_sales_rows
)

Original sales-line rows: 130402
Integrated sales rows: 130402
Difference: 0


### **13. Validate sales amount**

In [17]:
if (
    'unit_price' in sales_orders_lines_df.columns
    and
    'quantity' in sales_orders_lines_df.columns
):

    sales_orders_lines_df[
        'line_total'
    ] = (
        sales_orders_lines_df[
            'unit_price'
        ]
        *
        sales_orders_lines_df[
            'quantity'
        ]
    )

    print(
        "Calculated sales-line total:"
    )

    print(
        sales_orders_lines_df[
            'line_total'
        ].sum()
    )

Calculated sales-line total:
25447392560


### **14. Compare Sales Total**

In [20]:
# Compare Sales Header totals with Sales Lines totals

header_total = sales_orders_header_df['grand_total'].sum()

line_total = (
    sales_orders_lines_df['line_grand_total'].sum()
    if 'line_grand_total' in sales_orders_lines_df.columns
    else np.nan
)

print(f"Sales Header Grand Total: {header_total:,.2f}")
print(f"Sales Lines Grand Total:  {line_total:,.2f}")

if not pd.isna(line_total):

    difference = header_total - line_total

    print(f"Difference:               {difference:,.2f}")

Sales Header Grand Total: 32,424,754,555.80
Sales Lines Grand Total:  32,424,754,555.80
Difference:               0.00


### **15. Create validation scorecard**

In [19]:
print("=" * 70)
print("WEEK 2 DATA VALIDATION SCORECARD")
print("=" * 70)

print(
    f"Cleaned datasets checked: "
    f"{len(validation_datasets)}"
)

print(
    f"Integrated sales rows: "
    f"{len(sales_integrated_df):,}"
)

print(
    f"Integrated purchase rows: "
    f"{len(purchase_integrated_df):,}"
)

print(
    "\nTotal duplicate rows:",
    validation_duplicates_df[
        'Duplicate Rows'
    ].sum()
)

print(
    "Total missing cells:",
    validation_missing_df[
        'Missing Cells'
    ].sum()
)

print("\nValidation completed.")

WEEK 2 DATA VALIDATION SCORECARD
Cleaned datasets checked: 6
Integrated sales rows: 130,402
Integrated purchase rows: 155,495

Total duplicate rows: 0
Total missing cells: 17788

Validation completed.


**Save Validation Report.**

In [21]:
os.makedirs(
    '/content/week-02-validation',
    exist_ok=True
)

validation_rows.to_csv(
    '/content/week-02-validation/'
    'row_validation.csv',
    index=False
)

validation_duplicates_df.to_csv(
    '/content/week-02-validation/'
    'duplicate_validation.csv',
    index=False
)

validation_missing_df.to_csv(
    '/content/week-02-validation/'
    'missing_validation.csv',
    index=False
)

numeric_validation_df.to_csv(
    '/content/week-02-validation/'
    'numeric_validation.csv',
    index=False
)

date_validation_df.to_csv(
    '/content/week-02-validation/'
    'date_validation.csv',
    index=False
)

print("Validation reports saved.")

Validation reports saved.
